In [50]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import glob
import datetime


In [90]:
dir = 'output/estacoes'

estacao1 = 'BRASILIA'
estacao2 = 'SALVADOR'
estacao3 = 'MANAUS'
estacao4 = 'BELO HORIZONTE (PAMPULHA)'
estacao5 = 'PORTO ALEGRE - JARDIM BOTANICO'

ano1 = 2022
ano2 = 2023
ano3 = 2024

In [145]:
brasilia = glob.glob(f'output/estacoes/*{estacao1}*.CSV')
salvador = glob.glob(f'output/estacoes/*{estacao2}*.CSV')
manaus = glob.glob(f'output/estacoes/*{estacao3}*.CSV')
belo = glob.glob(f'output/estacoes/*{estacao4}*.CSV')
porto = glob.glob(f'output/estacoes/*{estacao5}*.CSV')

In [170]:
#selecione a estação
files = belo

### LIMPEZA

In [171]:
dados = []

for file in files:
  path = file
  print(path)
  df = pd.read_csv(path, delimiter=";", encoding='latin1', skiprows=8,usecols=['Data','Hora UTC','PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'])
  dados.append(df)

output/estacoes\INMET_SE_MG_A521_BELO HORIZONTE (PAMPULHA)_01-01-2022_A_31-12-2022.CSV
output/estacoes\INMET_SE_MG_A521_BELO HORIZONTE (PAMPULHA)_01-01-2023_A_31-12-2023.CSV
output/estacoes\INMET_SE_MG_A521_BELO HORIZONTE (PAMPULHA)_01-01-2024_A_31-12-2024.CSV


In [172]:
ds = pd.concat(dados)
ds

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"
0,2022/01/01,0000 UTC,",6"
1,2022/01/01,0100 UTC,"1,6"
2,2022/01/01,0200 UTC,",2"
3,2022/01/01,0300 UTC,",2"
4,2022/01/01,0400 UTC,1
...,...,...,...
8779,2024/12/31,1900 UTC,0
8780,2024/12/31,2000 UTC,0
8781,2024/12/31,2100 UTC,0
8782,2024/12/31,2200 UTC,0


In [173]:
ds = ds.rename(columns={'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao', 'Data': 'data', 'Hora UTC': 'hora'})
ds = ds.dropna()
ds

,data,hora,precipitacao
0,2022/01/01,0000 UTC,",6"
1,2022/01/01,0100 UTC,"1,6"
2,2022/01/01,0200 UTC,",2"
3,2022/01/01,0300 UTC,",2"
4,2022/01/01,0400 UTC,1
...,...,...,...
8779,2024/12/31,1900 UTC,0
8780,2024/12/31,2000 UTC,0
8781,2024/12/31,2100 UTC,0
8782,2024/12/31,2200 UTC,0


In [174]:
# limpa o " UTC" e garante formato consistente
ds['hora'] = ds['hora'].str.replace(' UTC', '').str.strip()

# cria coluna datetime combinando data e hora
ds['datetime'] = pd.to_datetime(
    ds['data'] + ' ' + ds['hora'],
    format='%Y/%m/%d %H%M',  # ou '%d/%m/%Y %H%M', dependendo do formato real
    errors='coerce'
)

# define o novo índice
ds = ds.set_index('datetime')
ds = ds.drop(['data', 'hora'],axis=1)

In [175]:
# substitui vírgulas por pontos e converte para float
ds['precipitacao'] = (ds['precipitacao'].astype(str).str.replace(',', '.', regex=False).astype(float))
ds_diario = ds.resample('D').sum(numeric_only=True)
print(ds_diario.head())

            precipitacao
datetime                
2022-01-01          31.0
2022-01-02           9.6
2022-01-03          14.8
2022-01-04          19.6
2022-01-05           8.8


### salvando o arquivo


In [176]:
parte = files[0].split('_')[4]  # depende da estrutura do nome
print(parte)

BELO HORIZONTE (PAMPULHA)


In [177]:
ds_diario.to_csv(f'{dir}/{parte}.csv')